# Michigan Traders: Module 4
# The Research Environment & Working with Financial Data  ·  *ANSWER KEY*

**Series:** MAT Education · QuantConnect Core
**Level:** Intermediate (builds on Modules 1–3)
**Format:** **Guided + Your-Turn**, with self-checking exercises you can run on your own laptop.

---

Module 3 taught you to *write* an algorithm. Before you write one, though, you should **research** it: pull the data, look at it, measure it, and decide whether the idea has any merit. That is what QuantConnect's **Research Environment** is for.

The Research Environment is a Jupyter notebook running on QuantConnect's servers with one extra object available: **`QuantBook`**. It gives you the same institutional data your backtest sees, but interactively: no backtest loop, no orders, just data in a DataFrame. Everything you learned in Module 2 about pandas applies directly.

By the end of this module you will be able to pull multi-asset price history, reshape it, turn it into returns, measure risk, and screen a small universe. The exact workflow that precedes every strategy you will build in Modules 5–9.

## How to use this notebook

This module runs in **two places**, and every code cell is labelled:

| Cell type | Where it runs | What to do |
|---|---|---|
| 🟢 **Local cell** | your laptop's Jupyter | Run it. Output is already baked in so you can read along. |
| 🔵 **QC cell** | QuantConnect Research | Copy into a QuantConnect research notebook. It will *not* run locally. `QuantBook` only exists on QC. |

To keep you practising without an internet connection, the local cells work on a **synthetic price panel shaped exactly like real `qb.history()` output**: same index, same column names, same `dtype`s. Every pandas move you make on it is the move you would make on QC.

The loop per topic is the usual one:

1. **Read** the explanation.
2. **Study** the worked example and its output.
3. **✏️ Your turn**: fill in the `# TODO`.
4. **Run the self-check**: it prints `✅ Correct!` or tells you exactly what is off.

Answers live in `04_Research_Environment_and_Financial_Data_SOLUTIONS.ipynb`. Try first.

### Setup: a stand-in for `qb.history()`

The next cell builds our practice panel. Read it carefully, because the *shape* it produces is the whole point of this module.

We simulate four US ETFs/stocks driven by a shared **market factor** plus asset-specific noise:

| Ticker | Exposure to the market factor | Reads as |
|---|---|---|
| `SPY` | 1.0 | the market itself |
| `AAPL` | 1.2 | a high-beta large-cap |
| `XLE` | 0.9 | an energy sector ETF |
| `TLT` | −0.3 | long bonds, which tend to move *against* equities |

Building the data this way (rather than as four independent random walks) gives us a realistic **correlation structure**. That is what makes the correlation and pairs work later in this notebook mean something.

The RNG is seeded, so your numbers will match the comments exactly.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.precision", 4)

rng = np.random.default_rng(7)
dates = pd.bdate_range("2022-01-03", periods=504)      # 504 business days ~ 2 trading years

# One shared market factor, then per-asset loadings on it (beta) + idiosyncratic noise.
market = rng.normal(0.0004, 0.009, len(dates))
specs = {                       # ticker: (beta on market, idio vol, start price)
    "SPY":  (1.0, 0.003, 400.0),
    "AAPL": (1.2, 0.012, 175.0),
    "XLE":  (0.9, 0.011,  70.0),
    "TLT":  (-0.3, 0.006, 100.0),
}

frames = []
for ticker, (beta, idio, p0) in specs.items():
    daily = beta * market + rng.normal(0.0, idio, len(dates))
    close = p0 * np.exp(np.cumsum(daily))              # compound returns into a price path
    intraday = rng.normal(0.0, 0.004, len(dates))
    frame = pd.DataFrame(
        {
            "close":  close,
            "high":   close * (1 + np.abs(intraday)),
            "low":    close * (1 - np.abs(intraday)),
            "open":   close * (1 - intraday),
            "volume": rng.integers(1_000_000, 9_000_000, len(dates)).astype(float),
        },
        index=pd.MultiIndex.from_product([[ticker], dates], names=["symbol", "time"]),
    )
    frames.append(frame)

history = pd.concat(frames)
print("shape:", history.shape, " = (4 tickers x 504 days, 5 columns)")
history.head(3)

shape: (2016, 5)  = (4 tickers x 504 days, 5 columns)


close      high       low      open     volume
symbol time                                                         
SPY    2022-01-03  401.1508  401.7640  400.5375  401.7640  4963739.0
       2022-01-04  401.9071  402.6082  401.2060  401.2060  4318883.0
       2022-01-05  400.0315  400.1294  399.9336  399.9336  5099128.0

That `history` object is our stand-in for what `qb.history(...)` hands back. Notice three things in the output above:

1. The index has **two levels**, `symbol` and `time`.
2. The columns are the five OHLCV fields, in alphabetical order (`close, high, low, open, volume`), exactly as QuantConnect returns them.
3. All four tickers are stacked **vertically** in one long table, not side by side.

Point 3 surprises everybody the first time. Section 4 explains why and how to reshape it.

## 1. What the Research Environment actually is

You have now met two QuantConnect objects. They are siblings, and it is worth being precise about the difference:

| | `QCAlgorithm` (Module 3) | `QuantBook` (this module) |
|---|---|---|
| Where it lives | `main.py` in an algorithm project | a notebook in the same project |
| Execution model | event-driven loop (`on_data` per bar) | you run cells, in any order |
| Can place orders | **yes** (`set_holdings`, `market_order`) | **no**, research only |
| Sees the future | no, strictly point-in-time | yes, you have the whole series at once |
| Main job | *execute* a strategy | *decide whether a strategy is worth executing* |

They share the same data library and the same method names wherever it makes sense: `add_equity` and `history` work on both.

> ⚠️ **The one danger of research.** In `on_data` your algorithm physically cannot see tomorrow's price. In research it can. The whole DataFrame is sitting in memory. Every look-ahead bias bug in quant finance starts here. Module 6 covers this properly; for now, just notice that the freedom is real and so is the risk.

### Creating a QuantBook

One line, at the top of a QC research notebook. This is a 🔵 QC cell. It will not run locally.

In [ ]:
# 🔵 QC cell, QuantConnect Research Environment only
qb = QuantBook()

# Subscribe exactly as you did in initialize() in Module 3.
# add_equity returns an Equity object; .symbol gives the Symbol handle.
spy = qb.add_equity("SPY", Resolution.DAILY).symbol
aapl = qb.add_equity("AAPL", Resolution.DAILY).symbol
tlt = qb.add_equity("TLT", Resolution.DAILY).symbol

print(spy, aapl, tlt)

Everything in that cell is a method you already know from Module 3 (`add_equity`, `Resolution.DAILY`, `.symbol`). The only new name is `QuantBook()` itself. That is deliberate: research and algorithm code are the *same API*, so what you learn in one transfers to the other.

## 2. `qb.history()`: pulling data

`history` is the workhorse. It has three call forms you will use constantly.

**Form A, one symbol, N bars back from now:**

```python
qb.history(symbol, 500, Resolution.DAILY)
```

**Form B: several symbols at once** (pass a list; this is the efficient way):

```python
qb.history([spy, aapl, tlt], 500, Resolution.DAILY)
```

**Form C: an explicit date range** (reproducible research; prefer this once you know your study window):

```python
qb.history([spy, aapl], datetime(2022, 1, 1), datetime(2024, 1, 1), Resolution.DAILY)
```

| Argument | Meaning |
|---|---|
| `symbol` / `[symbols]` | what to fetch. A list returns all of them in one DataFrame. |
| `periods` *or* `start, end` | how far back, a bar count, or two `datetime` objects. |
| `resolution` | `Resolution.DAILY`, `.HOUR`, `.MINUTE`, same enum as Module 3. |

A bar count of `500` at `Resolution.DAILY` means 500 *trading* days, roughly two calendar years.

In [ ]:
# 🔵 QC cell, the three call forms
from datetime import datetime

# A: single symbol, last 500 daily bars
one = qb.history(spy, 500, Resolution.DAILY)

# B: several symbols in one request (preferred - one round trip)
many = qb.history([spy, aapl, tlt], 500, Resolution.DAILY)

# C: an explicit, reproducible window
window = qb.history([spy, aapl, tlt],
                    datetime(2022, 1, 3), datetime(2024, 1, 1),
                    Resolution.DAILY)

print("single:", one.shape)
print("multi :", many.shape)
print("window:", window.shape)

### ✏️ Your turn: write the history call

Write a single `qb.history` call that fetches **AAPL, SPY and XLE**, **750 daily bars**, and stores the result in `panel`.

Assume `aapl`, `spy` and `xle` Symbol handles already exist. This is a 🔵 QC cell, so there is no self-check. Compare it against the answer key.

In [ ]:
# 🔵 QC cell, no local self-check for this one
panel = qb.history([aapl, spy, xle], 750, Resolution.DAILY)

print(panel.shape)

## 3. Anatomy of the returned DataFrame

When you pass a **list** of symbols, `history` returns a DataFrame with a **MultiIndex**: a two-level row index of `(symbol, time)`.

Module 2 gave you a DataFrame with one index (dates) and one column per ticker, a **wide** layout. QuantConnect hands you the opposite: a **long** layout, all tickers stacked, with the ticker recorded in the index instead of the column name.

Neither is better. Long is efficient for storage and for adding assets; wide is what you want for correlations and portfolio math. You need to move between them fluently, so let us look at the long form first.

In [2]:
print("index type   :", type(history.index).__name__)
print("index names  :", history.index.names)
print("n levels     :", history.index.nlevels)
print("columns      :", list(history.columns))
print("dtypes       :")
print(history.dtypes)

index type   : MultiIndex
index names  : ['symbol', 'time']
n levels     : 2
columns      : ['close', 'high', 'low', 'open', 'volume']
dtypes       :
close     float64
high      float64
low       float64
open      float64
volume    float64
dtype: object


`.index.levels` shows the distinct values available at each level. Level 0 is the symbols, level 1 is the timestamps.

In [3]:
print("symbols in the panel:", list(history.index.levels[0]))
print("first 3 dates       :", list(history.index.levels[1][:3].date))
print("last date           :", history.index.levels[1][-1].date())
print("rows per symbol     :", len(history) // len(history.index.levels[0]))

symbols in the panel: ['AAPL', 'SPY', 'TLT', 'XLE']
first 3 dates       : [datetime.date(2022, 1, 3), datetime.date(2022, 1, 4), datetime.date(2022, 1, 5)]
last date           : 2023-12-07
rows per symbol     : 504


> 💡 **On real QuantConnect data**, level 0 holds `Symbol` *objects*, not plain strings. They print like `SPY R735QTJ8P4TH` (ticker plus a unique security identifier). Indexing with the plain string `"SPY"` still works, because QuantConnect's Symbol type compares equal to its ticker. Our synthetic panel uses plain strings, which behave the same way for everything we do here.

## 4. Getting one symbol out

Three ways, in increasing order of precision.

**`.loc[ticker]`**: the simplest. Selecting on the outer index level returns that symbol's rows *with the symbol level dropped*, leaving a normal single-index DataFrame indexed by time. This is the one you will use most.

In [4]:
spy_bars = history.loc["SPY"]

print("type :", type(spy_bars).__name__)
print("index:", type(spy_bars.index).__name__, "->", spy_bars.index.name)
print("shape:", spy_bars.shape)
spy_bars.head(3)

type : DataFrame
index: DatetimeIndex -> time
shape: (504, 5)


,close,high,low,open,volume
time,,,,,
2022-01-03,401.1508,401.7640,400.5375,401.7640,4963739.0
2022-01-04,401.9071,402.6082,401.2060,401.2060,4318883.0
2022-01-05,400.0315,400.1294,399.9336,399.9336,5099128.0


Notice what just happened: `spy_bars` is exactly the kind of DataFrame Module 2 worked with, a `DatetimeIndex` and OHLCV columns. Every `.loc`, `.iloc`, `rolling`, `resample` and `pct_change` skill from Module 2 now applies unchanged.

**One column for one symbol**: chain a column selection to get a plain Series.

In [5]:
spy_close = history.loc["SPY", "close"]

print("type:", type(spy_close).__name__)
print("name:", spy_close.name)
print(spy_close.head(3))

type: Series
name: close
time
2022-01-03    401.1508
2022-01-04    401.9071
2022-01-05    400.0315
Freq: B, Name: close, dtype: float64


**`.xs()`**: the explicit version, short for *cross-section*. Use it when you want to slice on a level **by name**, which is clearer and works on either level.

`history.xs("SPY", level="symbol")` says "give me the rows where the `symbol` level equals SPY". Same result as `.loc["SPY"]`, but it says out loud which level it is slicing.

In [6]:
by_loc = history.loc["SPY"]
by_xs = history.xs("SPY", level="symbol")

print("same result:", by_loc.equals(by_xs))

# .xs really shines on the INNER level - all four tickers on one date:
one_day = history.xs("2023-06-15", level="time")
one_day

same result: True


,close,high,low,open,volume
symbol,,,,,
SPY,308.6914,309.9052,307.4776,307.4776,6567918.0
AAPL,163.3980,164.0695,162.7266,164.0695,2382119.0
XLE,84.8576,85.2468,84.4684,85.2468,3966255.0
TLT,88.2416,88.3792,88.1040,88.3792,5004008.0


That last table is a **cross-section**: one row per ticker, all on the same day. Cross-sectional slicing is the foundation of ranking strategies (Module 3's momentum capstone ranked a universe on a single date, and Modules 8–9 will do far more of it).

### ✏️ Your turn: pull one ticker out of the panel

From `history`, extract **AAPL's closing price series** into a variable called `aapl_close`.

It should be a pandas **Series** indexed by date (not a DataFrame), containing 504 values.

In [7]:
aapl_close = history.loc["AAPL", "close"]

print(type(aapl_close).__name__, aapl_close.shape)
aapl_close.head(3)

Series (504,)


time
2022-01-03    176.1107
2022-01-04    174.2017
2022-01-05    174.5454
Freq: B, Name: close, dtype: float64

In [8]:
expected = history.loc["AAPL", "close"]
assert isinstance(aapl_close, pd.Series), "aapl_close should be a Series, not a DataFrame"
assert len(aapl_close) == 504, f"expected 504 rows, got {len(aapl_close)}"
assert np.allclose(aapl_close.values, expected.values), "values do not match AAPL's close"
print("✅ Correct!  First close:", round(float(aapl_close.iloc[0]), 2),
      "| Last close:", round(float(aapl_close.iloc[-1]), 2))

✅ Correct!  First close: 176.11 | Last close: 124.82


## 5. Long → wide: `unstack`

For portfolio work you want a table with **one column per ticker** and one row per date, the Module 2 layout. `unstack` does exactly that: it takes an index level and turns its values into columns.

```
history["close"].unstack(level=0)
       │                      └── which index level becomes the columns (0 = symbol)
       └── pick ONE field first, so the result is 2-D
```

You unstack a **Series** (one field), not the whole 5-column DataFrame, otherwise you end up with a confusing two-level *column* index.

In [9]:
closes = history["close"].unstack(level=0)

print("shape:", closes.shape, " = (504 days, 4 tickers)")
print("columns:", list(closes.columns))
closes.head(3)

shape: (504, 4)  = (504 days, 4 tickers)
columns: ['AAPL', 'SPY', 'TLT', 'XLE']


symbol,AAPL,SPY,TLT,XLE
time,,,,
2022-01-03,176.1107,401.1508,100.4073,69.6775
2022-01-04,174.2017,401.9071,100.1660,70.9500
2022-01-05,174.5454,400.0315,99.8264,68.8747


Now it looks like Module 2's `prices` table, and every trick from that notebook works. You can also unstack by name instead of position, which reads better:

In [10]:
same = history["close"].unstack(level="symbol")
print("identical:", same.equals(closes))

# The reverse trip, for completeness: wide -> long
back_to_long = closes.stack()
print("stacked back to long:", back_to_long.shape, type(back_to_long).__name__)

identical: True
stacked back to long: (2016,) Series


> 🧠 **Rule of thumb.** Do your *filtering* in long form (it is one `.loc` away), then `unstack` the single field you care about and do your *math* in wide form.

### ✏️ Your turn: build a wide volume table

Build `volumes`: a wide DataFrame of daily **volume**, one column per ticker, one row per date, using the same `unstack` pattern.

Then compute `avg_volume`, a Series of each ticker's **mean** daily volume.

In [11]:
volumes = history["volume"].unstack(level="symbol")

avg_volume = volumes.mean()

avg_volume

symbol
AAPL    4.9799e+06
SPY     4.9442e+06
TLT     4.8935e+06
XLE     4.8011e+06
dtype: float64

In [12]:
exp_vol = history["volume"].unstack(level="symbol")
assert isinstance(volumes, pd.DataFrame), "volumes should be a DataFrame"
assert volumes.shape == (504, 4), f"expected (504, 4), got {volumes.shape}"
assert sorted(volumes.columns) == ["AAPL", "SPY", "TLT", "XLE"], "columns should be the four tickers"
assert np.allclose(volumes.sort_index(axis=1).values, exp_vol.sort_index(axis=1).values), \
    "volume values do not match"
assert isinstance(avg_volume, pd.Series) and len(avg_volume) == 4, \
    "avg_volume should be a Series with one entry per ticker"
assert np.allclose(sorted(avg_volume.values), sorted(exp_vol.mean().values)), \
    "avg_volume values are off"
print("✅ Correct!  Busiest ticker by average volume:", avg_volume.idxmax())

✅ Correct!  Busiest ticker by average volume: AAPL


## 6. Prices → returns, on a whole panel

Module 2 taught `pct_change()` on a single Series. On a wide DataFrame it does the same thing **column by column**, which is exactly what you want: each ticker's return computed against its own previous price.

The first row is always `NaN` (there is no day before the first day), so `.dropna()` immediately after is the standard idiom.

In [13]:
rets = closes.pct_change().dropna()

print("prices:", closes.shape, "-> returns:", rets.shape, "(one row lost to the NaN)")
rets.head(3)

prices: (504, 4) -> returns: (503, 4) (one row lost to the NaN)


symbol,AAPL,SPY,TLT,XLE
time,,,,
2022-01-04,-0.0108,0.0019,-0.0024,0.0183
2022-01-05,0.0020,-0.0047,-0.0034,-0.0292
2022-01-06,0.0074,-0.0132,-0.0006,0.0029


Sanity-check the numbers before trusting them. Daily equity returns should be small. A few tenths of a percent typically, with the odd 2–3% day.

In [14]:
summary = pd.DataFrame({
    "mean_daily": rets.mean(),
    "std_daily":  rets.std(),
    "min":        rets.min(),
    "max":        rets.max(),
})
summary

,mean_daily,std_daily,min,max
symbol,,,,
AAPL,-5.6221e-04,0.0156,-0.0502,0.0404
SPY,-7.4654e-04,0.0092,-0.0297,0.0263
TLT,-2.6899e-04,0.0063,-0.0177,0.0250
XLE,5.6139e-06,0.0135,-0.0441,0.0440


Read that table: `TLT` has the smallest daily swings (it is bonds), `AAPL` the largest (high beta plus its own noise). That matches how we built the data, which is a good sign that the pipeline is doing what we think it is.

**Log returns** work the same way, using the `np.log` / `diff` pattern from Module 1:

In [15]:
log_rets = np.log(closes).diff().dropna()

print("simple vs log, first 3 SPY values:")
comparison = pd.DataFrame({"simple": rets["SPY"].head(3), "log": log_rets["SPY"].head(3)})
comparison

simple vs log, first 3 SPY values:


,simple,log
time,,
2022-01-04,0.0019,0.0019
2022-01-05,-0.0047,-0.0047
2022-01-06,-0.0132,-0.0133


They are nearly identical at daily frequency. As Module 1 noted, `log(1 + r) ≈ r` for small `r`, and the difference only starts to matter when you compound over long horizons.

### ✏️ Your turn: daily returns and a cumulative curve

1. Build `daily_rets`, simple daily returns of `closes`, with the leading `NaN` row dropped.
2. Build `equity`, the cumulative growth of \$1 in each ticker, i.e. `(1 + r).cumprod()`.

`equity` should start near 1.0 and have the same shape as `daily_rets`.

In [16]:
daily_rets = closes.pct_change().dropna()

equity = (1 + daily_rets).cumprod()

equity.tail(3)

symbol,AAPL,SPY,TLT,XLE
time,,,,
2023-12-05,0.6832,0.6687,0.8614,0.9504
2023-12-06,0.6924,0.6650,0.8702,0.9429
2023-12-07,0.7087,0.6723,0.8647,0.9581


In [17]:
exp_r = closes.pct_change().dropna()
exp_e = (1 + exp_r).cumprod()
assert daily_rets.shape == (503, 4), f"daily_rets should be (503, 4), got {daily_rets.shape}"
assert not daily_rets.isna().any().any(), "daily_rets still contains NaN - did you dropna()?"
assert np.allclose(daily_rets.sort_index(axis=1).values, exp_r.sort_index(axis=1).values), \
    "return values are off"
assert np.allclose(equity.sort_index(axis=1).values, exp_e.sort_index(axis=1).values), \
    "equity curve values are off - remember (1 + r).cumprod()"
final = equity.iloc[-1].sort_values(ascending=False)
print("✅ Correct!  Best performer over the sample:", final.index[0],
      "at", round(float(final.iloc[0]), 3), "x")

✅ Correct!  Best performer over the sample: XLE at 0.958 x


## 7. Annualizing: turning daily numbers into comparable ones

A daily standard deviation of `0.011` means nothing to most people. Annualizing puts it on the scale everyone quotes.

Module 1 introduced the √252 rule; here it is again, applied to a whole panel at once:

| Quantity | Formula | Why |
|---|---|---|
| Annualized volatility | `daily_std * sqrt(252)` | variance scales with time, so std scales with √time |
| Annualized return (arithmetic) | `daily_mean * 252` | simple, fine for comparison |
| Annualized return (compound / CAGR) | `(final / initial) ** (252 / n_days) - 1` | what you actually earned |
| Sharpe ratio | `ann_return / ann_vol` | return per unit of risk (risk-free assumed 0) |

252 is the conventional number of US trading days in a year.

In [18]:
ann_vol = rets.std() * np.sqrt(252)
ann_ret = rets.mean() * 252
sharpe = ann_ret / ann_vol

stats = pd.DataFrame({"ann_return": ann_ret, "ann_vol": ann_vol, "sharpe": sharpe})
stats

,ann_return,ann_vol,sharpe
symbol,,,
AAPL,-0.1417,0.2481,-0.5711
SPY,-0.1881,0.1463,-1.2857
TLT,-0.0678,0.1004,-0.6752
XLE,0.0014,0.2139,0.0066


Now the numbers are readable: volatilities in the teens-to-twenties percent for equities, mid-single-digits for TLT.

A word on the **Sharpe ratio** column. Sharpe is return per unit of risk, and it is the single most quoted number in systematic trading. Over a two-year sample of simulated data these values are noisy. Do not read anything into the exact figures. What matters is the *method*, which is identical to what you will apply to backtest results in Module 6.

Here is the compound (CAGR) version, which is the honest one to quote for actual performance:

In [19]:
n_days = len(closes)
cagr = (closes.iloc[-1] / closes.iloc[0]) ** (252 / n_days) - 1

pd.DataFrame({"arithmetic_ann": ann_ret, "compound_cagr": cagr})

,arithmetic_ann,compound_cagr
symbol,,
AAPL,-0.1417,-0.1581
SPY,-0.1881,-0.1800
TLT,-0.0678,-0.0701
XLE,0.0014,-0.0212


The two disagree, and always will: the arithmetic mean of returns exceeds the compound growth rate whenever returns are volatile. That gap is *volatility drag*, and it grows with the square of volatility. AAPL, the most volatile name, shows the biggest gap.

### ✏️ Your turn: annualized risk table

Build `risk_table`, a DataFrame indexed by ticker with exactly two columns:

- `ann_vol`: annualized volatility from **`daily_rets`** (√252 rule)
- `ann_ret`: annualized arithmetic return (daily mean × 252)

Then set `riskiest` to the ticker with the highest annualized volatility.

In [20]:
risk_table = pd.DataFrame({
    "ann_vol": daily_rets.std() * np.sqrt(252),
    "ann_ret": daily_rets.mean() * 252,
})

riskiest = risk_table["ann_vol"].idxmax()

print(riskiest)
risk_table

AAPL


,ann_vol,ann_ret
symbol,,
AAPL,0.2481,-0.1417
SPY,0.1463,-0.1881
TLT,0.1004,-0.0678
XLE,0.2139,0.0014


In [21]:
exp_v = daily_rets.std() * np.sqrt(252)
exp_r2 = daily_rets.mean() * 252
assert isinstance(risk_table, pd.DataFrame), "risk_table should be a DataFrame"
assert set(risk_table.columns) == {"ann_vol", "ann_ret"}, \
    f"columns should be ann_vol and ann_ret, got {list(risk_table.columns)}"
assert np.allclose(risk_table["ann_vol"].sort_index().values, exp_v.sort_index().values), \
    "ann_vol is off - did you multiply by sqrt(252)?"
assert np.allclose(risk_table["ann_ret"].sort_index().values, exp_r2.sort_index().values), \
    "ann_ret is off - daily mean times 252"
assert riskiest == exp_v.idxmax(), f"riskiest should be {exp_v.idxmax()}"
print("✅ Correct!  Riskiest name:", riskiest,
      "at", f"{float(exp_v.max()):.1%}", "annualized vol")

✅ Correct!  Riskiest name: AAPL at 24.8% annualized vol


## 8. Rolling statistics across the panel

Module 2 introduced `.rolling(window)`. On a wide DataFrame it applies per column, so one line gives you every ticker's moving average.

The key detail to remember: the first `window - 1` rows are `NaN`, because a 60-day average needs 60 observations before it can produce anything.

In [22]:
ma60 = closes.rolling(60).mean()

print("first non-NaN row is at position:", ma60["SPY"].notna().argmax(), "(0-indexed)")
ma60.tail(3)

first non-NaN row is at position: 59 (0-indexed)


symbol,AAPL,SPY,TLT,XLE
time,,,,
2023-12-05,132.0532,289.2036,85.0699,76.3647
2023-12-06,131.7520,288.6465,85.0624,76.1300
2023-12-07,131.5036,288.1583,85.0392,75.9122


**Rolling volatility** is the same idea applied to returns, and it is how you *see* changing market regimes. Annualize it so the y-axis means something.

In [23]:
roll_vol = rets.rolling(60).std() * np.sqrt(252)

print("60-day annualized volatility, last observation:")
roll_vol.iloc[-1]

60-day annualized volatility, last observation:


symbol
AAPL    0.2643
SPY     0.1561
TLT     0.1124
XLE     0.2367
Name: 2023-12-07 00:00:00, dtype: float64

**Rolling correlation between two assets** takes a slightly different form: call `.rolling(...).corr(other_series)` on one Series and pass the other.

In [24]:
spy_aapl_corr = rets["SPY"].rolling(60).corr(rets["AAPL"])
spy_tlt_corr = rets["SPY"].rolling(60).corr(rets["TLT"])

print("60-day rolling correlation with SPY (last 3 observations)")
pd.DataFrame({"AAPL": spy_aapl_corr, "TLT": spy_tlt_corr}).tail(3)

60-day rolling correlation with SPY (last 3 observations)


,AAPL,TLT
time,,
2023-12-05,0.6388,-0.6350
2023-12-06,0.6209,-0.6277
2023-12-07,0.6335,-0.6338


SPY-AAPL correlation sits high and positive; SPY-TLT sits negative. That is the factor structure we built in the setup cell showing up in the statistics, which is exactly the kind of confirmation you look for when validating a research pipeline against data you understand.

### ✏️ Your turn: rolling Sharpe

Compute `roll_sharpe`, SPY's **60-day rolling annualized Sharpe ratio** from `rets`.

For each window: annualized mean (`mean × 252`) divided by annualized volatility (`std × √252`). Leave the leading `NaN` values in place.

Then set `best_sharpe` to the maximum value the series reaches.

In [25]:
spy_r = rets["SPY"]
roll_sharpe = (spy_r.rolling(60).mean() * 252) / (spy_r.rolling(60).std() * np.sqrt(252))

best_sharpe = roll_sharpe.max()

print(round(float(best_sharpe), 3))
roll_sharpe.tail(3)

2.589


time
2023-12-05   -2.7967
2023-12-06   -3.1421
2023-12-07   -2.6950
Freq: B, Name: SPY, dtype: float64

In [26]:
_r = rets["SPY"]
_exp = (_r.rolling(60).mean() * 252) / (_r.rolling(60).std() * np.sqrt(252))
assert isinstance(roll_sharpe, pd.Series), "roll_sharpe should be a Series"
assert len(roll_sharpe) == len(_r), "roll_sharpe should be the same length as the return series"
assert roll_sharpe.isna().sum() == 59, \
    f"expected 59 leading NaNs from a 60-day window, got {roll_sharpe.isna().sum()}"
assert np.allclose(roll_sharpe.dropna().values, _exp.dropna().values), \
    "values are off - annualize the mean by 252 and the std by sqrt(252)"
assert np.isclose(float(best_sharpe), float(_exp.max())), "best_sharpe should be the series max"
print("✅ Correct!  Peak 60-day rolling Sharpe:", round(float(best_sharpe), 2))

✅ Correct!  Peak 60-day rolling Sharpe: 2.59


## 9. The correlation matrix

`DataFrame.corr()` (Module 2, section 8) gives every pairwise correlation in one call. Always run it on **returns**, never on prices. Two unrelated assets that both drift upward will show a correlation near 1.0 on prices, which tells you nothing.

In [27]:
corr = rets.corr()
corr

symbol,AAPL,SPY,TLT,XLE
symbol,,,,
AAPL,1.0000,0.6134,-0.2074,0.4070
SPY,0.6134,1.0000,-0.3146,0.5648
TLT,-0.2074,-0.3146,1.0000,-0.2017
XLE,0.4070,0.5648,-0.2017,1.0000


Read the matrix: the diagonal is 1.0 by definition, and it is symmetric. SPY-AAPL and SPY-XLE are strongly positive; every TLT pairing is negative. A negative-correlation asset is valuable in a portfolio because it cushions drawdowns, which is the whole argument for holding bonds alongside equities.

To find the most-correlated *pair* you have to ignore the diagonal. The standard trick is to mask it out, then find the max.

In [28]:
masked = corr.where(~np.eye(len(corr), dtype=bool))   # NaN on the diagonal
pair = masked.stack().idxmax()                        # (row, col) of the largest value

print("most correlated pair:", pair, "->", round(float(masked.stack().max()), 3))
print("least correlated pair:", masked.stack().idxmin(), "->",
      round(float(masked.stack().min()), 3))

most correlated pair: ('AAPL', 'SPY') -> 0.613
least correlated pair: ('SPY', 'TLT') -> -0.315


`np.eye(n, dtype=bool)` builds an identity matrix of `True` on the diagonal (Module 1, section 2 covered `np.eye`). The `~` inverts it, and `.where()` keeps values only where the mask is `True`, writing `NaN` elsewhere. Then `.stack()` flattens the matrix to a Series indexed by `(row, col)` so `idxmax` can name the winning pair.

This is precisely the screen you run before building a pairs trade, which is where Module 7 picks up.

### ✏️ Your turn: the correlation screen

Using `daily_rets`:

1. Build `corr_matrix`, the 4×4 correlation matrix.
2. Set `spy_tlt_corr_value` to the correlation between **SPY and TLT** (a single float).
3. Set `diversifier` to the ticker whose **average correlation with the other three** is lowest, the best diversifier in the group. (Mask the diagonal first, then take a column mean.)

In [29]:
corr_matrix = daily_rets.corr()

spy_tlt_corr_value = corr_matrix.loc["SPY", "TLT"]

off_diagonal = corr_matrix.where(~np.eye(len(corr_matrix), dtype=bool))
diversifier = off_diagonal.mean().idxmin()

print(diversifier, round(float(spy_tlt_corr_value), 3))
corr_matrix

TLT -0.315


symbol,AAPL,SPY,TLT,XLE
symbol,,,,
AAPL,1.0000,0.6134,-0.2074,0.4070
SPY,0.6134,1.0000,-0.3146,0.5648
TLT,-0.2074,-0.3146,1.0000,-0.2017
XLE,0.4070,0.5648,-0.2017,1.0000


In [30]:
_c = daily_rets.corr()
_off = _c.where(~np.eye(len(_c), dtype=bool))
assert isinstance(corr_matrix, pd.DataFrame) and corr_matrix.shape == (4, 4), \
    "corr_matrix should be a 4x4 DataFrame"
assert np.allclose(np.diag(corr_matrix.values), 1.0), "the diagonal of a correlation matrix is 1.0"
assert np.isclose(float(spy_tlt_corr_value), float(_c.loc["SPY", "TLT"])), \
    "spy_tlt_corr_value should be corr_matrix.loc['SPY', 'TLT']"
assert diversifier == _off.mean().idxmin(), \
    f"the best diversifier here is {_off.mean().idxmin()} - mask the diagonal before averaging"
print("✅ Correct!  SPY-TLT correlation:", round(float(spy_tlt_corr_value), 3),
      "| best diversifier:", diversifier)

✅ Correct!  SPY-TLT correlation: -0.315 | best diversifier: TLT


## 10. Changing frequency: `resample`

Module 2 introduced `resample`. In research you use it constantly, because many signals are monthly even when the data is daily.

The rule that trips people up: **resample prices with `.last()`, resample returns with a compounding aggregation.** Taking the mean of prices inside a month is meaningless for a strategy that trades at month-end.

Remember the pandas 3.0 alias: `"ME"` = month-end (`"M"` was removed).

In [31]:
monthly_px = closes.resample("ME").last()

print("daily:", closes.shape, "-> monthly:", monthly_px.shape)
monthly_px.head(3)

daily: (504, 4) -> monthly: (24, 4)


symbol,AAPL,SPY,TLT,XLE
time,,,,
2022-01-31,165.0007,370.4955,96.5425,67.6684
2022-02-28,136.2117,340.1137,99.4963,63.6146
2022-03-31,150.5360,346.5133,98.9317,70.9333


Now monthly returns. There are two correct routes, and they agree:

In [32]:
# Route 1: percentage change of month-end prices
monthly_ret_a = monthly_px.pct_change().dropna()

# Route 2: compound the daily returns inside each month
monthly_ret_b = (1 + rets).resample("ME").prod() - 1

print("routes agree:",
      np.allclose(monthly_ret_a.values, monthly_ret_b.iloc[1:].values, atol=1e-10))
monthly_ret_a.head(3)

routes agree: True


symbol,AAPL,SPY,TLT,XLE
time,,,,
2022-02-28,-0.1745,-0.0820,0.0306,-0.0599
2022-03-31,0.1052,0.0188,-0.0057,0.1150
2022-04-30,-0.0518,-0.0061,-0.0034,0.0386


Route 2 (`(1 + r).resample("ME").prod() - 1`) is the one to remember, because it works even when you only have a return series and no prices, which is the usual situation once you are analysing a backtest's output.

### ✏️ Your turn: monthly view

From `closes`:

1. Build `month_end`, month-end closing prices (use `.last()`).
2. Build `monthly_returns`, monthly simple returns, `NaN` row dropped.
3. Set `best_month` to the **date label** of AAPL's single best month.

In [33]:
month_end = closes.resample("ME").last()

monthly_returns = month_end.pct_change().dropna()

best_month = monthly_returns["AAPL"].idxmax()

print(best_month)
monthly_returns.head(3)

2023-02-28 00:00:00


symbol,AAPL,SPY,TLT,XLE
time,,,,
2022-02-28,-0.1745,-0.0820,0.0306,-0.0599
2022-03-31,0.1052,0.0188,-0.0057,0.1150
2022-04-30,-0.0518,-0.0061,-0.0034,0.0386


In [34]:
_me = closes.resample("ME").last()
_mr = _me.pct_change().dropna()
assert month_end.shape == _me.shape, f"month_end should be {_me.shape}, got {month_end.shape}"
assert np.allclose(month_end.sort_index(axis=1).values, _me.sort_index(axis=1).values), \
    "month_end values are off - use .resample('ME').last()"
assert not monthly_returns.isna().any().any(), "monthly_returns still has NaN - dropna()"
assert np.allclose(monthly_returns.sort_index(axis=1).values, _mr.sort_index(axis=1).values), \
    "monthly_returns values are off"
assert best_month == _mr["AAPL"].idxmax(), "best_month should come from idxmax on AAPL"
print("✅ Correct!  AAPL's best month:", str(best_month)[:7],
      "at", f"{float(_mr['AAPL'].max()):.1%}")

✅ Correct!  AAPL's best month: 2023-02 at 13.8%


## 11. Indicators in the Research Environment

Module 3 created indicators inside `initialize` and read them bar by bar in `on_data`. Research gives you a shortcut: **`qb.indicator(...)`** runs an indicator over a whole history in one call and hands back a DataFrame you can plot or join.

```python
qb.indicator(indicator_object, symbol, periods, resolution)
```

This is enormously useful for deciding parameters *before* you commit them to an algorithm. You can compare a 20-day and a 50-day moving average over ten years in seconds, instead of running ten backtests.

In [ ]:
# 🔵 QC cell, indicator history in research
from QuantConnect.Indicators import SimpleMovingAverage, RelativeStrengthIndex

sma_20 = qb.indicator(SimpleMovingAverage(20), spy, 500, Resolution.DAILY)
rsi_14 = qb.indicator(RelativeStrengthIndex(14), spy, 500, Resolution.DAILY)

print(sma_20.tail(3))
print(rsi_14.tail(3))

The returned object is an ordinary pandas DataFrame indexed by time, so you can join it straight onto your price table and study the signal:

In [ ]:
# 🔵 QC cell, join an indicator onto prices and inspect the signal
prices = qb.history(spy, 500, Resolution.DAILY)["close"].unstack(level=0)

study = prices.join(sma_20["simplemovingaverage"].rename("sma20"))
study["above_ma"] = study.iloc[:, 0] > study["sma20"]

print("share of days above the 20-day MA:", round(study["above_ma"].mean(), 3))
study.tail(3)

> 💡 You can reproduce any of this locally with pandas. A simple moving average *is* `closes.rolling(20).mean()`. The value of `qb.indicator` is that it uses **the identical indicator implementation your algorithm will use**, including its warm-up behaviour, so research and live results agree. Module 5 goes deep on indicators.

## 12. Missing data across tickers

Real panels are ragged. A stock that IPO'd in 2021 has no data in 2019, so its rows are simply absent from the long frame, and after `unstack` they become `NaN`.

Module 2's alignment rules apply, but the decision is yours to make explicitly. Let us manufacture the situation and look at the options.

In [35]:
ragged = closes.copy()
ragged.loc[ragged.index[:120], "XLE"] = np.nan     # pretend XLE listed 120 days late

print("NaN count per ticker:")
print(ragged.isna().sum())

NaN count per ticker:
symbol
AAPL      0
SPY       0
TLT       0
XLE     120
dtype: int64


**Option A, `dropna()`**: keep only dates where *every* ticker has data. Safe, and the right default for correlation and portfolio work, but you lose history for every asset because one was late.

In [36]:
complete = ragged.dropna()
print("rows:", len(ragged), "->", len(complete), "(everything before XLE's start is gone)")

rows: 504 -> 384 (everything before XLE's start is gone)


**Option B, analyse pairwise**: `.corr()` already does this by default, using whatever overlap each pair has. Note the differing sample sizes hiding behind those numbers.

In [37]:
print("pairwise correlations on the ragged panel (returns):")
ragged.pct_change().corr()

pairwise correlations on the ragged panel (returns):


symbol,AAPL,SPY,TLT,XLE
symbol,,,,
AAPL,1.0000,0.6134,-0.2074,0.4311
SPY,0.6134,1.0000,-0.3146,0.5992
TLT,-0.2074,-0.3146,1.0000,-0.2180
XLE,0.4311,0.5992,-0.2180,1.0000


**Option C, forward-fill** (`.ffill()`): carry the last known price forward. This is right for a *holiday gap* in an otherwise-listed asset, and **wrong** for a pre-IPO gap. It would invent prices for a stock that did not trade yet, and any backtest on it would be fiction.

> ⚠️ Never `ffill` across a period when a security did not exist. The most common way a beginner's backtest reports a fantastic Sharpe ratio is by trading an asset that had no market.

### ✏️ Your turn: clean the ragged panel

Using `ragged`:

1. Build `clean`. The panel restricted to rows where **all four** tickers have a price.
2. Set `first_complete_date` to `clean`'s first index label.
3. Set `rows_lost` to how many rows were dropped versus `ragged`.

In [38]:
clean = ragged.dropna()

first_complete_date = clean.index[0]

rows_lost = len(ragged) - len(clean)

print(first_complete_date, "| rows lost:", rows_lost)

2022-06-20 00:00:00 | rows lost: 120


In [39]:
_clean = ragged.dropna()
assert len(clean) == len(_clean), f"clean should have {len(_clean)} rows, got {len(clean)}"
assert not clean.isna().any().any(), "clean still contains NaN"
assert first_complete_date == _clean.index[0], "first_complete_date should be clean's first index label"
assert rows_lost == 120, f"120 rows should be lost, got {rows_lost}"
print("✅ Correct!  Panel starts", str(first_complete_date)[:10], "after dropping", rows_lost, "rows")

✅ Correct!  Panel starts 2022-06-20 after dropping 120 rows


## 🏁 Mini-challenge: a reusable research summary

Every research notebook you write this year will start by answering the same question: *what do these assets look like?* Write it once as a function and you will use it for the rest of the curriculum.

### ✏️ Your turn: a research summary function

Write `research_summary(price_df)` that takes a **wide** price DataFrame and returns a DataFrame indexed by ticker with exactly these four columns, in this order:

| column | definition |
|---|---|
| `ann_return` | daily mean return × 252 |
| `ann_vol` | daily std × √252 |
| `sharpe` | `ann_return / ann_vol` |
| `max_drawdown` | most negative value of `equity / equity.cummax() - 1`, where `equity = (1 + r).cumprod()` |

Max drawdown should come out **negative** (it is a loss). Module 1 built this same statistic in NumPy; this is the pandas version.

In [40]:
def research_summary(price_df):
    r = price_df.pct_change().dropna()

    ann_return = r.mean() * 252
    ann_vol = r.std() * np.sqrt(252)
    sharpe = ann_return / ann_vol

    equity = (1 + r).cumprod()
    max_drawdown = (equity / equity.cummax() - 1).min()

    return pd.DataFrame({
        "ann_return": ann_return,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "max_drawdown": max_drawdown,
    })


research_summary(closes)

,ann_return,ann_vol,sharpe,max_drawdown
symbol,,,,
AAPL,-0.1417,0.2481,-0.5711,-0.3178
SPY,-0.1881,0.1463,-1.2857,-0.3362
TLT,-0.0678,0.1004,-0.6752,-0.2065
XLE,0.0014,0.2139,0.0066,-0.3323


In [41]:
out = research_summary(closes)
_r = closes.pct_change().dropna()
_eq = (1 + _r).cumprod()
_exp = pd.DataFrame({
    "ann_return": _r.mean() * 252,
    "ann_vol": _r.std() * np.sqrt(252),
    "sharpe": (_r.mean() * 252) / (_r.std() * np.sqrt(252)),
    "max_drawdown": (_eq / _eq.cummax() - 1).min(),
})
assert list(out.columns) == ["ann_return", "ann_vol", "sharpe", "max_drawdown"], \
    f"columns must be in the stated order, got {list(out.columns)}"
assert sorted(out.index) == ["AAPL", "SPY", "TLT", "XLE"], "index should be the tickers"
for col in _exp.columns:
    assert np.allclose(out[col].sort_index().values, _exp[col].sort_index().values), \
        f"the {col} column is off"
assert (out["max_drawdown"] <= 0).all(), "max_drawdown should be negative"

# and it must work on any wide price frame, not just this one
small = research_summary(closes[["SPY", "TLT"]])
assert small.shape == (2, 4), "the function should work on any wide price DataFrame"
print("✅ Correct!  Worst drawdown in the panel:", out["max_drawdown"].idxmin(),
      "at", f"{float(out['max_drawdown'].min()):.1%}")

✅ Correct!  Worst drawdown in the panel: SPY at -33.6%


## Putting it together on QuantConnect

Here is the whole module as one research cell. This is the shape of every research notebook you will write from here on: subscribe, fetch, reshape, measure, decide.

In [ ]:
# 🔵 QC cell, a complete research workflow
from datetime import datetime

qb = QuantBook()

tickers = ["SPY", "AAPL", "XLE", "TLT"]
symbols = [qb.add_equity(t, Resolution.DAILY).symbol for t in tickers]

# 1. fetch a reproducible window
history = qb.history(symbols, datetime(2022, 1, 3), datetime(2024, 1, 1), Resolution.DAILY)

# 2. reshape long -> wide on the field we care about
closes = history["close"].unstack(level=0)
closes.columns = [str(c).split()[0] for c in closes.columns]   # Symbol -> plain ticker

# 3. measure
rets = closes.pct_change().dropna()
stats = pd.DataFrame({
    "ann_return": rets.mean() * 252,
    "ann_vol": rets.std() * np.sqrt(252),
})
stats["sharpe"] = stats["ann_return"] / stats["ann_vol"]

# 4. decide
print(stats.sort_values("sharpe", ascending=False))
print()
print("correlation matrix:")
print(rets.corr())

One line there is new: `closes.columns = [str(c).split()[0] for c in closes.columns]`. On real QuantConnect data the columns are `Symbol` objects that print as `SPY R735QTJ8P4TH`; splitting on whitespace and keeping the first piece recovers the plain ticker so your tables are readable. You will paste that line into most of your research notebooks.

## Cheat sheet

| Task | Code |
|---|---|
| Start research | `qb = QuantBook()` |
| Subscribe | `sym = qb.add_equity("SPY", Resolution.DAILY).symbol` |
| History, N bars | `qb.history(sym, 500, Resolution.DAILY)` |
| History, many symbols | `qb.history([s1, s2], 500, Resolution.DAILY)` |
| History, date range | `qb.history([s1], start_dt, end_dt, Resolution.DAILY)` |
| One symbol out of the panel | `history.loc["SPY"]` or `history.xs("SPY", level="symbol")` |
| One date, all symbols | `history.xs("2023-06-15", level="time")` |
| Long → wide | `history["close"].unstack(level="symbol")` |
| Wide → long | `closes.stack()` |
| Returns | `closes.pct_change().dropna()` |
| Log returns | `np.log(closes).diff().dropna()` |
| Equity curve | `(1 + rets).cumprod()` |
| Annualized vol | `rets.std() * np.sqrt(252)` |
| Annualized return | `rets.mean() * 252` |
| CAGR | `(p[-1] / p[0]) ** (252 / n) - 1` |
| Max drawdown | `(eq / eq.cummax() - 1).min()` |
| Rolling stat | `closes.rolling(60).mean()` |
| Rolling correlation | `a.rolling(60).corr(b)` |
| Correlation matrix | `rets.corr()` |
| Mask the diagonal | `corr.where(~np.eye(len(corr), dtype=bool))` |
| Month-end prices | `closes.resample("ME").last()` |
| Compound to monthly | `(1 + rets).resample("ME").prod() - 1` |
| Indicator over history | `qb.indicator(SimpleMovingAverage(20), sym, 500, Resolution.DAILY)` |
| Drop incomplete rows | `panel.dropna()` |

## Stretch goals

Bring these to the next meeting:

1. **Rebuild the panel with a 5-year window** and recompute the correlation matrix. Is the SPY-TLT relationship stable, or does it flip in some sub-periods? Slice the sample in half and compare.
2. **Add a fifth asset** with a beta of `0.0` (pure noise, no market exposure) and confirm the correlation screen identifies it as the best diversifier.
3. **Compare arithmetic and compound annual returns** across all four tickers and plot the gap against annualized volatility. Does the volatility-drag relationship look quadratic?
4. **Write `rolling_beta(asset, market, window)`** returning a rolling regression beta, using `cov / var`. Check that SPY's beta against itself is exactly 1.0 everywhere.
5. **On QuantConnect**, pull ten years of SPY daily data and compare `qb.indicator(SimpleMovingAverage(20), ...)` against `closes.rolling(20).mean()`. Where do they differ, and why? (Look at the first few rows.)

## What's next

**Module 5: Indicators, Signals & Strategy Design Patterns** takes the indicators you have only glanced at so far and makes them the centre of the work: how QuantConnect constructs them, the difference between automatic and manual updates, warm-up, and the handful of signal patterns (crossover, threshold, breakout, filter) that nearly every systematic strategy is built from.

**Official docs:**
- [Research Environment overview](https://www.quantconnect.com/docs/v2/research-environment)
- [Research: historical data](https://www.quantconnect.com/docs/v2/research-environment/datasets/equity)
- [`History` requests in algorithms](https://www.quantconnect.com/docs/v2/writing-algorithms/historical-data/history-requests)
- [Indicators in research](https://www.quantconnect.com/docs/v2/research-environment/indicators)
- [pandas: MultiIndex](https://pandas.pydata.org/docs/user_guide/advanced.html)

*MAT Education · QuantConnect Core · Module 4.*